# 07 — DFS Constraint

**Repo:** `decoherence-free-sensing`  
**Lab report:** *Decoherence-Free Subspaces Specify Robust Quantum Sensing*  
**Paper:** Kaubruegger *et al.*, *Physical Review X* **16**, 021052 (2026), DOI: `10.1103/ksyh-mb4s`

Notebook 00 established the leading specification:

\[
\text{reject common-mode noise first, then engineer entanglement.}
\]

Notebook 07 makes that rule explicit. A decoherence-free sensor is specified by keeping preparation, evolution, and measurement inside one fixed \(J_z^+\) sector, where common-mode phase noise acts only as a global phase.

## Engineering statement

A decoherence-free sensor is not specified by entanglement alone.

It is specified by compatibility with the fixed \(J_z^+\) sector:

\[
J_z^+\lvert \psi \rangle = m\lvert \psi \rangle.
\]

Inside that sector,

\[
U_\Phi\lvert\psi\rangle
= e^{-i\Phi J_z^+}\lvert\psi\rangle
= e^{-i\Phi m}\lvert\psi\rangle,
\]

so the common phase \(\Phi\) becomes global and does not change measurement probabilities.


## Notebook outputs

Running this notebook creates:

- `results/figures/07_dfs_sector_ladder_v2.png`
- `results/figures/07_common_phase_as_global_phase_v2.png`
- `results/figures/07_measurement_commutation_rule_v2.png`
- `results/figures/07_inside_vs_outside_dfs_v2.png`
- `results/figures/07_dfs_design_rule_summary_v2.png`
- `results/dfs_constraint_summary.json`
- `results/dfs_constraint_summary.csv`
- `results/decoherence_free_sensing_07_dfs_constraint_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})

FIGURE_PATHS = []

def save_figure(fig, filename):
    path = FIGURES / filename
    fig.savefig(path, bbox_inches="tight")
    FIGURE_PATHS.append(path)
    plt.show()
    return path


def add_box(ax, xy, width, height, title, body, facecolor="white", lw=1.8, title_size=15, body_size=11):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.05",
        linewidth=lw,
        edgecolor="black",
        facecolor=facecolor
    )
    ax.add_patch(box)
    ax.text(x + width/2, y + height*0.64, title,
            ha="center", va="center", fontsize=title_size, fontweight="bold")
    ax.text(x + width/2, y + height*0.34, body,
            ha="center", va="center", fontsize=body_size)
    return box


def add_arrow(ax, x0, y0, x1, y1):
    arr = FancyArrowPatch(
        (x0, y0), (x1, y1),
        arrowstyle="-|>", mutation_scale=18, linewidth=1.8
    )
    ax.add_patch(arr)
    return arr


## 1. Fixed \(J_z^+\) sectors

For two nodes, the common-mode phase is generated by

\[
J_z^+ = J_z^A + J_z^B.
\]

The Hilbert space splits into sectors labeled by the eigenvalue \(m\):

\[
J_z^+\lvert \psi_m\rangle = m\lvert \psi_m\rangle.
\]

Common phase only distinguishes sectors. Inside one fixed sector it becomes a global phase.


In [ ]:
# Toy sector ladder for four atoms: two in node A, two in node B.
# Arrows mark spin-up/spin-down basis states. The central m=0 sector is highlighted.

sectors = {
    2: ["↑↑ | ↑↑"],
    1: ["↑↑ | ↑↓", "↑↑ | ↓↑", "↑↓ | ↑↑", "↓↑ | ↑↑"],
    0: ["↑↑ | ↓↓", "↑↓ | ↑↓", "↑↓ | ↓↑", "↓↑ | ↑↓", "↓↑ | ↓↑", "↓↓ | ↑↑"],
    -1: ["↑↓ | ↓↓", "↓↑ | ↓↓", "↓↓ | ↑↓", "↓↓ | ↓↑"],
    -2: ["↓↓ | ↓↓"],
}

fig, ax = plt.subplots(figsize=(12.8, 6.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, r"Hilbert Space Splits into Fixed $J_z^+$ Sectors",
        ha="center", va="center", fontsize=25)

ys = {2: 0.78, 1: 0.62, 0: 0.46, -1: 0.30, -2: 0.14}
for m, y in ys.items():
    ax.text(0.085, y, rf"$m={m}$", ha="right", va="center", fontsize=15)
    ax.plot([0.15, 0.85], [y, y], linewidth=1.5, alpha=0.55)
    text = "   ".join(sectors[m])
    if m == 0:
        # thicker, light-blue central DFS highlight
        rect = FancyBboxPatch((0.18, y-0.035), 0.66, 0.07,
                              boxstyle="round,pad=0.01,rounding_size=0.012",
                              linewidth=2.8, edgecolor="black", facecolor="#eaf4ff")
        ax.add_patch(rect)
        ax.text(0.51, y, text, ha="center", va="center", fontsize=13)
        ax.text(0.86, y, "central DFS", ha="left", va="center", fontsize=14)
    else:
        ax.text(0.51, y, text, ha="center", va="center", fontsize=13, alpha=0.75)

ax.text(
    0.5, 0.035,
    r"Common phase only distinguishes sectors. Inside one fixed sector it becomes a global phase.",
    ha="center", va="center", fontsize=15
)

save_figure(fig, "07_dfs_sector_ladder_v2.png")


## 2. Common phase becomes global

The common-mode unitary is

\[
U_\Phi = e^{-i\Phi J_z^+}.
\]

If the state has support only in one fixed sector, then

\[
U_\Phi\lvert\psi_m\rangle
= e^{-i\Phi m}\lvert\psi_m\rangle.
\]

The phase factor is global. Measurement probabilities are unchanged by \(\Phi\).


In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 5.2))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.90, "Common Phase Becomes Global Inside One DFS Sector",
        ha="center", va="center", fontsize=24)

boxes = [
    (0.06, "common-mode noise", r"$U_\Phi=e^{-i\Phi J_z^+}$"),
    (0.31, "fixed sector", r"$J_z^+|\psi\rangle=m|\psi\rangle$"),
    (0.56, "global phase", r"$U_\Phi|\psi\rangle=e^{-i\Phi m}|\psi\rangle$"),
    (0.81, "probabilities", r"unchanged by $\Phi$"),
]

w, h, y = 0.18, 0.28, 0.45
for x, title, body in boxes:
    add_box(ax, (x, y), w, h, title, body, title_size=15, body_size=12)

for x in [0.24, 0.49, 0.74]:
    add_arrow(ax, x, y + h/2, x + 0.07, y + h/2)

ax.text(
    0.5, 0.18,
    r"Design rule: choose states, operations, and measurements that do not leak information out of the fixed $J_z^+$ sector.",
    ha="center", va="center", fontsize=15
)

save_figure(fig, "07_common_phase_as_global_phase_v2.png")


## 3. Measurement compatibility

A measurement is compatible with the DFS constraint when it does not mix different \(J_z^+\) sectors.

The rule is the commutator condition:

\[
[M,J_z^+] = 0.
\]

DFS-compatible observables preserve sector labels. Local single-spin observables such as \(\sigma_x^A\) generally leak out of a fixed sector because they flip only one node.


In [ ]:
# Basis for two spins: ↑↑, ↑↓, ↓↑, ↓↓.
# Assign Jz+ sector labels m = 1, 0, 0, -1 in arbitrary units.
basis = ["↑↑", "↑↓", "↓↑", "↓↓"]
mvals = np.array([1, 0, 0, -1])

# Compatible toy operator: nonzero only within same m sector.
M_compatible = np.zeros((4, 4))
M_compatible[1, 2] = 1
M_compatible[2, 1] = 1

# Incompatible local flip on node A: ↑↑ <-> ↓↑ and ↑↓ <-> ↓↓.
sigma_x_A = np.zeros((4, 4))
sigma_x_A[0, 2] = 1
sigma_x_A[2, 0] = 1
sigma_x_A[1, 3] = 1
sigma_x_A[3, 1] = 1

Jplus = np.diag(mvals)
comm_good = M_compatible @ Jplus - Jplus @ M_compatible
comm_bad = sigma_x_A @ Jplus - Jplus @ sigma_x_A

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.6))
fig.suptitle("Measurement Compatibility Is a Commutation Rule", fontsize=24, y=1.03)

images = []
for ax, mat, title in [
    (axes[0], np.abs(comm_good), r"compatible: $[M,J_z^+]=0$"),
    (axes[1], np.abs(comm_bad), r"leaks DFS sector: local $\sigma_x^A$"),
]:
    im = ax.imshow(mat, vmin=0, vmax=1)
    images.append(im)
    ax.set_title(title, fontsize=16, pad=8)
    ax.set_xticks(range(4), basis)
    ax.set_yticks(range(4), basis)
    ax.set_xlabel("basis state")
    ax.set_ylabel("basis state")

cbar = fig.colorbar(images[-1], ax=axes.ravel().tolist(), shrink=0.78)
cbar.set_label("absolute commutator entry")

save_figure(fig, "07_measurement_commutation_rule_v2.png")


## 4. Inside versus outside the DFS

Inside one fixed sector, the common phase is invisible to probabilities.

Across multiple sectors, the common phase creates sector-dependent relative phases:

\[
\lvert\psi\rangle = a\lvert m_1\rangle + b\lvert m_2\rangle
\quad\Rightarrow\quad
U_\Phi\lvert\psi\rangle
= ae^{-i\Phi m_1}\lvert m_1\rangle
+ be^{-i\Phi m_2}\lvert m_2\rangle.
\]

The relative phase changes with \(\Phi\), so common-mode noise becomes observable decoherence.


In [ ]:
Phi = np.linspace(0, 2*np.pi, 500)
inside = np.ones_like(Phi)
# Proxy for a coherence term between two sectors with Delta m = 2.
outside = np.cos(2 * Phi)

fig, ax = plt.subplots(figsize=(8.6, 5.8))
ax.plot(Phi, inside, linewidth=2.2, label=r"fixed $J_z^+$ sector")
ax.plot(Phi, outside, linewidth=2.2, label="multiple sectors")
ax.set_title("Inside DFS: Common Phase Invisible. Outside DFS: Relative Phase Moves.")
ax.set_xlabel(r"common phase $\Phi$")
ax.set_ylabel("normalized coherence proxy")
ax.grid(True, alpha=0.3)
ax.legend(loc="center right")

ax.annotate("global only", xy=(4.8, 1.0), xytext=(4.2, 0.55),
            arrowprops=dict(arrowstyle="->", linewidth=1.2), fontsize=12)
ax.annotate("relative phase", xy=(4.7, -1.0), xytext=(3.5, -0.7),
            arrowprops=dict(arrowstyle="->", linewidth=1.2), fontsize=12)

save_figure(fig, "07_inside_vs_outside_dfs_v2.png")


## 5. DFS design rule summary

The DFS constraint must be respected by the whole sensing protocol:

1. **Prepare** the state in one fixed \(J_z^+\) sector.
2. **Evolve** with operations that preserve \(J_z^+\).
3. **Measure** observables that commute with \(J_z^+\).
4. **Estimate** the differential phase generated by \(J_z^-\).

Then common-mode phase \(\Phi\) is rejected while differential phase \(\varphi\) remains measurable.


In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.90, "DFS Design Rule Summary", ha="center", va="center", fontsize=25)

boxes = [
    (0.06, "Prepare", r"fixed $J_z^+$"),
    (0.31, "Evolve", r"preserve $J_z^+$"),
    (0.56, "Measure", r"commute with $J_z^+$"),
    (0.81, "Estimate", r"use $J_z^-$"),
]

w, h, y = 0.18, 0.28, 0.48
for x, title, body in boxes:
    add_box(ax, (x, y), w, h, title, body, title_size=16, body_size=13)

for x in [0.24, 0.49, 0.74]:
    add_arrow(ax, x, y + h/2, x + 0.07, y + h/2)

ax.text(
    0.5, 0.27,
    r"Common-mode phase $\Phi$ is rejected only when the full protocol respects the DFS constraint.",
    ha="center", va="center", fontsize=15
)
ax.text(
    0.5, 0.17,
    r"Then entanglement can be optimized for differential phase $\varphi$ without amplifying shared noise.",
    ha="center", va="center", fontsize=15
)

save_figure(fig, "07_dfs_design_rule_summary_v2.png")


## 6. Context summary table

In [ ]:
summary = pd.DataFrame([
    {
        "item": "notebook",
        "value": "07_dfs_constraint"
    },
    {
        "item": "core_question",
        "value": "Why does a fixed J_z^+ sector reject common-mode phase noise?"
    },
    {
        "item": "common_phase_unitary",
        "value": "U_Phi = exp(-i Phi J_z^+)"
    },
    {
        "item": "dfs_condition",
        "value": "J_z^+ |psi> = m |psi>"
    },
    {
        "item": "global_phase_result",
        "value": "U_Phi |psi> = exp(-i Phi m) |psi>"
    },
    {
        "item": "measurement_rule",
        "value": "DFS-compatible measurements satisfy [M, J_z^+] = 0"
    },
    {
        "item": "operation_rule",
        "value": "Preparation and evolution must preserve the fixed J_z^+ sector"
    },
    {
        "item": "differential_signal",
        "value": "The useful phase is generated by J_z^- = J_z^A - J_z^B"
    },
    {
        "item": "engineering_statement",
        "value": "Choose states, operations, and measurements that reject shared noise before optimizing entanglement."
    },
])

summary_csv = RESULTS / "dfs_constraint_summary.csv"
summary_json = RESULTS / "dfs_constraint_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 7. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_07_dfs_constraint_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(FIGURES.glob("07_*_v2.png")):
        zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Wrote {zip_path}")
zip_path

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))


## 8. Next notebook

Suggested next notebook:

`13_phase_coordinates.ipynb`

Core goal:

- make common and differential coordinates explicit,
- simulate common-mode noise and differential readout,
- show why the differential coordinate is the admissible signal once \(\Phi\) has been rejected.
